# Chương 5 - Knowledge Base, Knowledge Graph và RAG

<span style='color:#c62828'><b>PHẦN CODE ĐÃ BỔ SUNG.</b> Notebook trình bày đầy đủ ingestion, kiểm tra schema và provenance, chunking, năm cấu hình B0–B4, answer có citation, request contract, benchmark, ablation, phân tích lỗi và triển khai API. Chạy tuần tự từ trên xuống.</span>

## 1. Khởi tạo

Mọi đường dẫn được suy ra từ thư mục dự án. Notebook dùng chung code trong `src/` với web để tránh một phiên bản demo khác phiên bản deploy.

In [1]:
from pathlib import Path
import inspect, json, sys
ROOT = Path.cwd()
if not (ROOT / 'knowledge').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('Project:', ROOT)

Project: C:\DATA\assignment3\tieuluan3


## 2. Nạp ba lớp dữ liệu

`corpus.json` là kho tài liệu có URL; `graph.json` là các triple có evidence; `benchmark.json` chỉ dùng để đánh giá sau retrieval. Hai CSV Diabetes và AAPL là dữ liệu quan sát của các chương trước, không bị trộn vào gold evidence.

In [2]:
corpus=json.loads((ROOT/'knowledge/corpus.json').read_text(encoding='utf-8'))
triples=json.loads((ROOT/'knowledge/graph.json').read_text(encoding='utf-8'))
queries=json.loads((ROOT/'knowledge/benchmark.json').read_text(encoding='utf-8'))
print('Documents:',len(corpus),'| Triples:',len(triples),'| Queries:',len(queries))
print('Domains:',{d:sum(x['domain']==d for x in corpus) for d in ['health','finance']})

Documents: 12 | Triples: 23 | Queries: 15
Domains: {'health': 6, 'finance': 6}


## 3. Kiểm tra schema, liên kết và provenance

Hàm dưới đây từ chối document thiếu trường, ID trùng, URL không phải HTTPS, triple có evidence treo và query có gold ID không tồn tại. Ba hash khóa đúng snapshot dùng cho kết quả.

In [3]:
from knowledge_pipeline import validate_knowledge_base
quality=validate_knowledge_base(corpus,triples,queries)
quality

{'documents': 12, 'triples': 23, 'queries': 15, 'domains': {'finance': 6, 'health': 6}, 'document_hash': 'af28a6ee38b629577161bd10bf86da7d50979a1776415aa93c81283f019758b2', 'graph_hash': 'af4a797c7dae393b9a337d0d842e482eaaddd54233cbc0f31c7da25e1ded3047', 'benchmark_hash': 'ae3c41b07ac0a30c1742149cb6aec9800b915e08701bebec0b6129e92b37bbfa', 'status': 'valid'}

### Code kiểm tra dữ liệu

In ra source thật đang được notebook và script thực nghiệm sử dụng.

In [4]:
print(inspect.getsource(validate_knowledge_base))

def validate_knowledge_base(documents: list[dict], triples: list[dict], queries: list[dict]) -> dict:
    """Reject missing schema fields, duplicate IDs and dangling provenance."""
    issues: list[str] = []
    document_ids = [row.get("id") for row in documents]
    if len(document_ids) != len(set(document_ids)):
        issues.append("Document id bị trùng.")
    for index, row in enumerate(documents):
        missing = REQUIRED_DOCUMENT_FIELDS - set(row)
        if missing:
            issues.append(f"Document {index} thiếu {sorted(missing)}.")
        if row.get("domain") not in ALLOWED_DOMAINS:
            issues.append(f"Document {row.get('id')} có domain không hợp lệ.")
        if not str(row.get("url", "")).startswith("https://"):
            issues.append(f"Document {row.get('id')} thiếu URL HTTPS.")
        if not str(row.get("text", "")).strip():
            issues.append(f"Document {row.get('id')} rỗng.")
    known = set(document_ids)
    triple_keys = set()
    for index, r

## 4. Ingestion và chunking

Tài liệu được cắt theo cửa sổ 45 token, overlap 8 token. Mỗi chunk giữ `document_id`, URL, ngày nguồn, vị trí token và SHA-256. Corpus nhỏ vẫn chạy được ở cấp document; artifact chunk chứng minh pipeline có thể mở rộng mà không mất nguồn.

In [5]:
from knowledge_pipeline import chunk_documents
chunks=chunk_documents(corpus,max_tokens=45,overlap=8)
print('Chunks:',len(chunks))
print(json.dumps(chunks[0],ensure_ascii=False,indent=2))

Chunks: 23
{
  "chunk_id": "health_overview::c000",
  "document_id": "health_overview",
  "domain": "health",
  "title": "WHO - Tổng quan đái tháo đường",
  "url": "https://www.who.int/news-room/fact-sheets/detail/diabetes",
  "as_of": "2024-11-14",
  "token_start": 0,
  "token_end": 45,
  "text": "Đái tháo đường là bệnh mạn tính xảy ra khi tuyến tụy không sản xuất đủ insulin hoặc cơ thể không sử dụng insulin hiệu quả. Tăng glucose kéo dài có thể gây tổn thương tim, mạch máu, mắt, thận và thần kinh. Type 1",
  "sha256": "1d212d45af03a81c0febfffab343adc4b670d9c399890d283547c25c994807ec"
}


## 5. Knowledge Graph và tính toàn vẹn evidence

Mỗi cạnh có `subject`, `relation`, `object`, `evidence`. Evidence phải là ID của document trong corpus, vì vậy một đường graph luôn truy ngược được về URL.

In [6]:
for t in triples[:10]: print(f"({t['subject']}) -[{t['relation']}]-> ({t['object']}) [{t['evidence']}]")

(đái tháo đường) -[có loại]-> (type 1) [health_overview]
(đái tháo đường) -[có loại]-> (type 2) [health_overview]
(đái tháo đường) -[có thể có dấu hiệu]-> (khát nhiều) [health_symptoms]
(đái tháo đường) -[có thể có dấu hiệu]-> (tiểu nhiều) [health_symptoms]
(đái tháo đường) -[có thể gây]-> (tổn thương thận) [health_overview]
(đái tháo đường) -[được đánh giá bằng]-> (A1C) [health_testing]
(đái tháo đường) -[được đánh giá bằng]-> (glucose huyết tương) [health_testing]
(type 2) -[có thể phòng ngừa bằng]-> (vận động thể lực) [health_prevention]
(type 2) -[có thể phòng ngừa bằng]-> (dinh dưỡng lành mạnh) [health_prevention]
(CDC Diabetes) -[được dùng bởi]-> (MLP) [health_dataset]


## 6. Khởi tạo chỉ mục

Constructor tạo thống kê BM25, ma trận TF-IDF, phép chiếu SVD tám chiều và adjacency list. Tất cả chạy offline bằng NumPy.

In [7]:
from knowledge_rag import KnowledgeRAG
engine=KnowledgeRAG(ROOT/'knowledge')
print('TF-IDF:',engine.tfidf.shape,'| LSA dense:',engine.dense.shape,'| corpus hash:',engine.corpus_sha256[:16])

TF-IDF: (12, 284) | LSA dense: (12, 8) | corpus hash: af28a6ee38b62957


## 7. B0 — BM25 sparse retrieval

BM25 dùng tần suất từ, IDF và chuẩn hóa độ dài. Đây là baseline lexical, nhanh và giải thích được.

In [8]:
print(inspect.getsource(KnowledgeRAG.bm25))
print(engine.bm25('xét nghiệm glucose và A1C')[:3])

    def bm25(self, query: str) -> list[tuple[str, float]]:
        q = tokenize(query)
        scores = []
        for doc, terms, dl in zip(self.documents, self.doc_tokens, self.lengths):
            tf, score = Counter(terms), 0.0
            for term in q:
                freq = tf.get(term, 0)
                if freq:
                    score += self.idf.get(term, 0) * freq * 2.5 / (freq + 1.5 * (.25 + .75 * dl / self.avgdl))
            if score > 0:
                scores.append((doc["id"], float(score)))
        return sorted(scores, key=lambda x: (-x[1], x[0]))

[('health_testing', 8.994423719810513), ('health_model', 3.046977747133819), ('health_overview', 1.5370952599461303)]


## 8. B1 — LSA dense retrieval

LSA chiếu TF-IDF qua SVD rồi tính cosine. Từ “dense” ở đây chỉ vector SVD dày, không phải embedding Transformer.

In [9]:
print(inspect.getsource(KnowledgeRAG.lsa))
print(engine.lsa('giá đầu phiên và khối lượng giao dịch')[:3])

    def lsa(self, query: str) -> list[tuple[str, float]]:
        q = self._query_vector(query) @ self.projection
        denom = np.linalg.norm(self.dense, axis=1) * np.linalg.norm(q)
        scores = (self.dense @ q) / np.maximum(denom, 1e-12)
        rows = [(d["id"], float(s)) for d, s in zip(self.documents, scores) if s > 1e-12]
        return sorted(rows, key=lambda x: (-x[1], x[0]))

[('stock_ohlcv', 0.9572930950178685), ('stock_baseline', 0.5706019748238813), ('stock_limits', 0.5489237701006153)]


## 9. B3 — graph retrieval

Entity linker dùng alias xác định; BFS duyệt tối đa `hops`. Hệ thống không đoán entity khi không có alias và không tạo cạnh mới lúc truy vấn.

In [10]:
print(inspect.getsource(KnowledgeRAG.graph))
ranked,paths=engine.graph('AAPL liên hệ Apple, OHLCV và RNN',hops=2)
print(ranked[:5]); print(json.dumps(paths[:3],ensure_ascii=False,indent=2))

    def graph(self, query: str, hops: int = 2) -> tuple[list[tuple[str, float]], list[dict]]:
        nq = " " + normalize(query) + " "
        seeds = [node for node in self.alias if f" {node} " in nq]
        queue = deque((s, 0) for s in seeds)
        seen = set(seeds)
        evidence, paths = defaultdict(float), []
        while queue:
            node, depth = queue.popleft()
            if depth >= hops:
                continue
            for nxt, triple in self.adj.get(node, []):
                score = 1 / (depth + 1)
                evidence[triple["evidence"]] += score
                paths.append({**triple, "hop": depth + 1})
                if nxt not in seen:
                    seen.add(nxt)
                    queue.append((nxt, depth + 1))
        ranked = sorted(evidence.items(), key=lambda x: (-x[1], x[0]))
        return ranked, paths

[('apple_ir', 3.5), ('stock_baseline', 3.0), ('stock_ohlcv', 3.0), ('stock_return', 3.0), ('stock_dataset', 2.0)]
[
  {
    "subj

## 10. B2/B4 — Reciprocal Rank Fusion

B2 hợp BM25 và LSA. B4 thêm KG. RRF dùng thứ hạng nên không cần coi điểm BM25, cosine và graph là cùng đơn vị.

In [11]:
print(inspect.getsource(KnowledgeRAG.rrf))
question='AAPL liên hệ với Apple, OHLCV và RNN như thế nào?'
comparison=engine.compare(question,top_k=3,hops=2)
for system,result in comparison.items(): print(system,result['method'],'->',[d['id'] for d in result['documents']])

    @staticmethod
    def rrf(rankings: list[tuple[list[tuple[str, float]], float]], constant: int = 60):
        scores = defaultdict(float)
        for ranking, weight in rankings:
            for rank, (doc_id, _) in enumerate(ranking, 1):
                scores[doc_id] += weight / (constant + rank)
        return sorted(scores.items(), key=lambda x: (-x[1], x[0]))

B0 BM25 -> ['stock_dataset', 'apple_ir', 'stock_limits']
B1 LSA -> ['apple_ir', 'stock_limits', 'stock_return']
B2 Hybrid -> ['apple_ir', 'stock_limits', 'stock_dataset']
B3 KG -> ['apple_ir', 'stock_baseline', 'stock_ohlcv']
B4 GraphRAG -> ['apple_ir', 'stock_dataset', 'stock_ohlcv']


## 11. Hợp hàm toàn pipeline

Với câu hỏi $q$: normalize → B0/B1/B3 → RRF → top-k → grounded answer → citations/abstention. `compare` khóa cùng input, top-k và corpus cho cả B0–B4.

In [12]:
answer=engine.answer('AAPL có baseline nào và RNN có phải khuyến nghị đầu tư không?',method='B4')
print(answer['answer'])
print('Abstained:',answer['abstained'],'| Sources:',[x['id'] for x in answer['citations']])
print('Request hash:',answer['request_sha256'])

Hệ thống từ chối đưa ra kết luận: Yêu cầu khuyến nghị mua hoặc bán tài sản.
Abstained: True | Sources: []
Request hash: 7be67d27fe6b3f662217dcb3599a757fb5aff2965628058f9faa3b3e20463ff8


## 12. Request contract và trường hợp biên

API chỉ nhận năm trường allowlist, giới hạn độ dài, top-k, số hop và domain. Input lạ bị từ chối thay vì âm thầm bỏ qua.

In [13]:
from knowledge_pipeline import validate_request
valid=validate_request({'question':'Dấu hiệu diabetes là gì?','method':'B4','top_k':3,'hops':2,'domain':'health'})
print(valid)
for invalid in [{}, {'question':'x'}, {'question':'AAPL','method':'unknown'}, {'question':'AAPL','top_k':100}]:
    try: validate_request(invalid)
    except ValueError as exc: print('REJECT:',exc)

{'question': 'Dấu hiệu diabetes là gì?', 'method': 'GraphRAG', 'top_k': 3, 'hops': 2, 'domain': 'health'}
REJECT: question phải dài từ 3 đến 500 ký tự.
REJECT: question phải dài từ 3 đến 500 ký tự.
REJECT: method phải thuộc ['B0', 'B1', 'B2', 'B3', 'B4'] hoặc ['BM25', 'LSA', 'Hybrid', 'KG', 'GraphRAG'].
REJECT: top_k phải là số nguyên từ 1 đến 10.


## 13. Grounded answer, citation và từ chối

Answer chỉ lấy câu từ tài liệu đã truy hồi. Câu yêu cầu giá chắc chắn, kê đơn hoặc khuyến nghị mua/bán bị từ chối và không gắn citation giả.

In [14]:
for q in ['Xét nghiệm nào đánh giá diabetes?','Giá AAPL ngày mai chắc chắn bằng bao nhiêu?','Quỹ đạo Voyager là gì?']:
    r=engine.answer(q,method='B4')
    print('\nQ:',q,'\n',r['answer'],'\nabstained=',r['abstained'],'citations=',[c['id'] for c in r['citations']])


Q: Xét nghiệm nào đánh giá diabetes? 
 Bác sĩ có thể dùng A1C, glucose huyết tương lúc đói, nghiệm pháp dung nạp glucose đường uống và glucose huyết tương ngẫu nhiên để đánh giá đái tháo đường. Score chưa được calibration cho sử dụng lâm sàng, không phân biệt type 1 và type 2, không thay thế xét nghiệm hoặc chẩn đoán. Dự báo từ mô hình RNN là kết quả thực nghiệm trên dữ liệu lịch sử, không phải cam kết về giá tương lai hoặc khuyến nghị mua bán. 
abstained= False citations= ['health_testing', 'health_model', 'stock_limits']

Q: Giá AAPL ngày mai chắc chắn bằng bao nhiêu? 
 Hệ thống từ chối đưa ra kết luận: Yêu cầu dự báo chắc chắn về tương lai. 
abstained= True citations= []

Q: Quỹ đạo Voyager là gì? 
 Hệ thống từ chối đưa ra kết luận: Câu hỏi nằm ngoài phạm vi kho tri thức hiện tại. 
abstained= True citations= []


## 14. Metric retrieval đầy đủ

Ngoài Recall và MRR, notebook tính Precision@3, nDCG@3, Complete@3, OOD abstention cùng p50/p95 latency. Gold chỉ dùng sau khi ranking đã tạo.

In [15]:
import pandas as pd
from IPython.display import display,Image
metrics=pd.read_csv(ROOT/'results/knowledge_rag/comparison.csv')
display(metrics.round(4))
display(Image(filename=ROOT/'results/knowledge_rag/retrieval_comparison.png'))

  System    Method  ...  Latency_p50_ms  Latency_p95_ms
0     B0      BM25  ...          1.5003          1.9083
1     B1       LSA  ...          1.5043          2.2068
2     B2    Hybrid  ...          1.3581          1.7240
3     B3        KG  ...          1.1758          1.7204
4     B4  GraphRAG  ...          1.8847          2.2979

[5 rows x 10 columns]
<IPython.core.display.Image object>


## 15. Ablation số hop và top-k

Ablation chỉ thay một nhóm tham số, giữ nguyên corpus và câu hỏi. `hops=0` loại đóng góp graph; top-k 1/3/5 cho thấy đánh đổi precision–coverage.

In [16]:
ablation=pd.read_csv(ROOT/'results/knowledge_rag/ablation.csv')
display(ablation.round(4).fillna(''))

    hops  top_k   MRR@1   MRR@3  ... Recall@5  nDCG@1  nDCG@3  nDCG@5
0      0      1  0.9167          ...           0.9167                
1      0      3          0.9583  ...                   0.9056        
2      0      5                  ...   0.9583                  0.9224
3      1      1  0.8333          ...           0.8333                
4      1      3          0.9167  ...                   0.9063        
5      1      5                  ...      1.0                  0.9246
6      2      1  0.8333          ...           0.8333                
7      2      3          0.9167  ...                   0.9063        
8      2      5                  ...      1.0                   0.926
9      3      1    0.75          ...             0.75                
10     3      3           0.875  ...                   0.8755        
11     3      5                  ...      1.0                  0.8953

[12 rows x 11 columns]


## 16. Phân tích lỗi ở mức query

Mỗi lỗi ghi system, query ID, gold bị thiếu và danh sách đã truy hồi. File prediction JSONL còn giữ score, answer, citation, latency và hash để tái tính metric.

In [17]:
errors=pd.read_csv(ROOT/'results/knowledge_rag/error_analysis.csv')
print('Error rows:',len(errors))
display(errors.head(12))

Error rows: 13
   system  ...            error_type
0      B0  ...  incomplete_retrieval
1      B0  ...  incomplete_retrieval
2      B1  ...  incomplete_retrieval
3      B1  ...  incomplete_retrieval
4      B1  ...  incomplete_retrieval
5      B2  ...  incomplete_retrieval
6      B2  ...  incomplete_retrieval
7      B3  ...  incomplete_retrieval
8      B3  ...  incomplete_retrieval
9      B3  ...  incomplete_retrieval
10     B3  ...  incomplete_retrieval
11     B4  ...  incomplete_retrieval

[12 rows x 7 columns]


## 17. Hai case study theo miền

Case sức khỏe giới hạn ở thông tin nguồn và nhắc không chẩn đoán. Case tài chính nối AAPL–OHLCV–RNN–baseline và từ chối biến dự báo thành lời khuyên đầu tư.

In [18]:
for domain,q in [('health','Type 2 phòng ngừa thế nào và nguồn tổng quan nói gì?'),('finance','Vì sao RNN AAPL cần so với Last value?')]:
    r=engine.answer(q,method='B4',domain=domain)
    print('\nDOMAIN',domain,'\n',r['answer'],'\n',[c['url'] for c in r['citations']])


DOMAIN health 
 Chế độ ăn lành mạnh, hoạt động thể lực thường xuyên, duy trì cân nặng phù hợp và tránh thuốc lá có thể giúp phòng ngừa hoặc làm chậm type 2. Type 1 đặc trưng bởi thiếu sản xuất insulin; type 2 liên quan cơ thể sử dụng insulin không hiệu quả. Type 2 có thể phát triển trong nhiều năm và một số người không nhận thấy triệu chứng. 
 ['https://www.who.int/news-room/fact-sheets/detail/diabetes', 'https://www.who.int/news-room/fact-sheets/detail/diabetes', 'https://www.cdc.gov/diabetes/signs-symptoms/']

DOMAIN finance 
 Mốc Last value dự báo giá phiên tiếp theo bằng giá Close gần nhất. Bài tạo cửa sổ 30 phiên và tám đặc trưng từ OHLCV để dự báo log-return phiên kế tiếp rồi đổi về giá USD. Đặc trưng chuỗi gồm log-return của Open, High, Low, Close, log Volume, độ lệch Close so với MA5 và MA20, cùng log tỷ lệ High/Low. 
 ['https://github.com/Vinhdiesel28/tieuluan3', 'https://github.com/tuananhtrieu1305/ISD_Assignment01/tree/main/A06/datasets', 'https://github.com/Vinhdiesel28/ti

## 18. Manifest và khả năng tái lập

Manifest ghi schema version, cấu hình chunk, hash và kích cỡ corpus/graph/benchmark/chunk. Đây là provenance của lần chạy, không phải mô tả nhập tay.

In [19]:
manifest=json.loads((ROOT/'results/knowledge_rag/pipeline_manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest,ensure_ascii=False,indent=2))

{
  "schema_version": "1.0.0",
  "chapter": 5,
  "systems": {
    "B0": "BM25",
    "B1": "LSA",
    "B2": "Hybrid",
    "B3": "KG",
    "B4": "GraphRAG"
  },
  "quality": {
    "documents": 12,
    "triples": 23,
    "queries": 15,
    "domains": {
      "finance": 6,
      "health": 6
    },
    "document_hash": "af28a6ee38b629577161bd10bf86da7d50979a1776415aa93c81283f019758b2",
    "graph_hash": "af4a797c7dae393b9a337d0d842e482eaaddd54233cbc0f31c7da25e1ded3047",
    "benchmark_hash": "ae3c41b07ac0a30c1742149cb6aec9800b915e08701bebec0b6129e92b37bbfa",
    "status": "valid"
  },
  "chunking": {
    "max_tokens": 45,
    "overlap": 8,
    "chunks": 23
  },
  "files": {
    "corpus": {
      "bytes": 6487,
      "sha256": "9c44261d165575692834ea815e1c1f5553b6625a3dd8e1f5443d487422cc885f"
    },
    "graph": {
      "bytes": 2542,
      "sha256": "2aefdee3ec64148adf9293e30a866fd2378a7e74536f7478fe193c635a1d81cd"
    },
    "benchmark": {
      "bytes": 2204,
      "sha256": "1d0a2b1214d2

## 19. Triển khai

`POST /api/rag` nhận request đúng schema và trả method, ranking, answer, paths, citations, abstention và hash. `POST /api/agent` dùng B4 như một công cụ trong hệ đa Agent. Cả hai không cần API key hoặc dịch vụ graph bên ngoài.

In [20]:
payload={'question':'AAPL gồm dữ liệu nào?','method':'B4','top_k':3,'hops':2,'domain':'finance'}
print('POST /api/rag')
print(json.dumps(payload,ensure_ascii=False,indent=2))

POST /api/rag
{
  "question": "AAPL gồm dữ liệu nào?",
  "method": "B4",
  "top_k": 3,
  "hops": 2,
  "domain": "finance"
}


## 20. Kết luận và giới hạn

So với phiên bản trước, code đã có ingestion/chunking, validation, provenance hash, B0–B4, request contract, grounded answer, abstention, metric mở rộng, ablation, prediction row-level và error analysis. Corpus 12 tài liệu vẫn nhỏ; kết quả không được suy rộng thành chất lượng trên MedQuAD, FinQA hoặc một hệ GraphRAG quy mô lớn.